# Python AI Agent · 15讲逐格实战

原理讲解：https://zbyloveyj.github.io/personal-growth-lab/academy/agent-lab/jupyter/

从上到下运行每格。默认离线，不需要API Key。所有DEMO记录为**虚构教学数据**，绝不代表真实科研论文。

## 1 · 环境安装与数据准备

**核心问题：**为什么先用虚构数据而不是联网搜索？

Jupyter Notebook 可以把说明文字、Python代码和运行结果放在同一个文件中。先使用可重复的虚构微生物研究记录，能够排除网络延迟、API权限和模型费用的干扰，验证每一步输入与输出。实验数据明确标记为教学模拟，不能当作真实论文证据。

**操作步骤：**
1. 运行环境检查，观察Python版本及工作目录
2. 加载DEMO_STUDIES，查看每一条记录的字段与数据类型
3. 修改一条教学记录，观察搜索结果如何改变

**预期输出：**能解释Notebook单元格的执行顺序，并打印一组标明DEMO的结构化记录。

**误区：**Notebook可能因为跳着运行产生变量未定义。遇到此情况应从第一格重新运行，不要把教学数据当成发表论文。

In [ ]:
import sys,json,sqlite3,re
from pathlib import Path
print('Python',sys.version.split()[0])
OUTPUT=Path('agent_lab_output');OUTPUT.mkdir(exist_ok=True)
STUDIES=[{'id':'DEMO01','disease':'MDD','microbe':'Faecalibacterium'},{'id':'DEMO02','disease':'BD','microbe':'Blautia'},{'id':'DEMO03','disease':'SZ','microbe':'Bacteroides'}]
print(json.dumps(STUDIES,ensure_ascii=False,indent=2))

## 2 · 工具调用与函数白名单

**核心问题：**Agent所谓的“调用工具”，到底调用了什么？

工具不是模型临时生成的魔法，而是开发者预先提供的、拥有明确输入输出契约的Python函数。以微生物研究助手为例，一个函数负责按疾病检索，另一个负责计算，外层的白名单决定允许调用哪些工具。加入参数校验，比直接执行模型生成的字符串安全得多。

**操作步骤：**
1. 运行安全计算器，比较允许的运算和被拒绝的表达式
2. 运行search_studies检索MDD和BD示例记录
3. 打开TOOL_REGISTRY，理解工具名与Python函数的映射关系

**预期输出：**能够解释工具名、参数、函数体、返回值四个部分，并在程序中追加一个只读工具。

**误区：**不应对用户输入直接使用eval、exec或未限制的Shell命令。

In [ ]:
def search_studies(query):
    aliases={'抑郁症':'MDD','双相':'BD','精神分裂':'SZ'}
    q=aliases.get(query,query).lower()
    return [r for r in STUDIES if q in json.dumps(r,ensure_ascii=False).lower()]
TOOLS={'search_studies':search_studies}
print(search_studies('抑郁症'))

## 3 · 建立第一个规则式Agent

**核心问题：**程序怎样知道先搜索还是先计算？

一个最小Agent由任务输入、动作选择、工具执行、观察结果和最终回答五个环节构成。这里先使用规则决策器，便于追踪它为什么选择某个函数。它能展示完整闭环，但并没有使用大语言模型，所以不能把输出称为自主推理或真正的模型决策。

**操作步骤：**
1. 把“查询抑郁症研究”交给规则式Agent
2. 观察决策器选择search_studies的记录
3. 把问题改成比较两种疾病，观察规则适用范围

**预期输出：**能顺着执行轨迹复述“问题→决策→工具→结果→回答”。

**误区：**规则覆盖范围有限；模型能力与程序规则必须清楚区分。

In [ ]:
def rule_agent(question):
    q=next((x for x in ('抑郁症','双相','精神分裂') if x in question),question)
    return {'decision':'search_studies','query':q,'result':search_studies(q)}
print(rule_agent('查询抑郁症教学记录'))

## 4 · Function Calling完整链路

**核心问题：**模型说“调用函数”，是谁真正执行了Python？

Function Calling是模型与程序之间的结构化约定。模型根据函数描述返回一个调用请求，宿主Python程序负责验证工具名称和参数、执行真实函数，再把结果交回模型。模型并不会凭空操作本机文件。理解这一点以后，就能看懂多个模型服务为什么可以复用相同工具代码。

**操作步骤：**
1. 读取模拟模型输出的JSON工具请求
2. 解析name和arguments，并通过执行器完成调用
3. 故意请求不存在的工具，观察程序如何拒绝

**预期输出：**能够画出模型、函数调用消息、Python执行器和工具结果的往返关系。

**误区：**模型建议的参数不是可信输入；必须在执行层再验证，不可直接反序列化成任意代码。

In [ ]:
call=json.loads('{"name":"search_studies","arguments":{"query":"BD"}}')
assert call['name'] in TOOLS
assert set(call['arguments'])=={'query'}
assert isinstance(call['arguments']['query'],str)
print(TOOLS[call['name']](**call['arguments']))

## 5 · 短期记忆与长期存储

**核心问题：**聊天记录为什么不等于长期记忆？

短期记忆是当前任务中有限的上下文与工具轨迹；长期记忆是明确保留的知识、偏好或经历，必须有独立存储、更新和删除策略。本实验使用本地JSON与SQLite分别演示会话上下文、持久化和安全查询。真实研究系统还要考虑隐私、权限、信息过期和跨用户隔离。

**操作步骤：**
1. 向AgentMemory写入两个对话回合
2. 保存示例偏好到本地JSON并重新读取
3. 使用SQLite参数化SQL插入与查询教学记录

**预期输出：**能说明内存中的变量、JSON文件和SQLite各自怎样保存状态。

**误区：**不能把敏感患者记录、API Key或个人信息直接写进公开GitHub仓库。

In [ ]:
class Memory:
    def __init__(self):self.history=[]
    def add(self,q,a):self.history.append({'question':q,'answer':a})
memory=Memory();memory.add('MDD','教学记录')
(OUTPUT/'memory.json').write_text(json.dumps(memory.history,ensure_ascii=False),encoding='utf-8')
with sqlite3.connect(OUTPUT/'agent.sqlite') as db:
    db.execute('CREATE TABLE IF NOT EXISTS notes(topic TEXT, note TEXT)')
    db.execute('INSERT INTO notes VALUES (?,?)',('MDD','仅供教学'))
    print(db.execute('SELECT * FROM notes WHERE topic=?',('MDD',)).fetchall())

## 6 · 推理、规划与逐步执行

**核心问题：**复杂任务为什么要先拆解？

“比较抑郁症与双相障碍证据”并不是一个单独函数，而是由确定疾病、分别检索、对齐字段、比较结果、形成结论等环节组成。规划器明确依赖顺序，再逐步调用工具，这种分解也便于定位错误。初学阶段采用固定规划流程；接入LLM后仍需对每个行动进行约束。

**操作步骤：**
1. 调用ResearchPlanner生成两疾病比较计划
2. 逐步执行计划并记录中间输出
3. 故意让一个步骤缺少参数，观察失败处理

**预期输出：**获得包含顺序动作和工具返回值的结构化执行轨迹。

**误区：**计划文本不是事实；每一步的结果必须有实际工具输出支撑。

In [ ]:
plan=[{'action':'search_studies','query':'MDD'},{'action':'search_studies','query':'BD'}]
trace=[]
for step in plan:
    result=TOOLS[step['action']](step['query'])
    trace.append({'step':step,'result':result})
print(json.dumps(trace,ensure_ascii=False,indent=2))

## 7 · RAG检索增强生成

**核心问题：**为什么不能直接让模型凭记忆回答科研问题？

RAG先把资料整理成文档片段，再根据问题检索证据，最后把检索到的片段交给生成模块组织回答。这里使用简单TF-IDF来演示索引与排序，而不是声称已经实现高性能语义搜索。重要的是理解：检索命中并不必然意味着证据支持答案，科研场景还需检查研究设计、效应方向和来源。

**操作步骤：**
1. 建立包含ID、标题、正文与来源的文档集合
2. 比较不同问题的检索排序与得分
3. 运行grounded_answer并检查每一句依据了哪些记录

**预期输出：**得到带教学来源ID的检索结果和保守回答。

**误区：**本节TF-IDF适合演示，不等于向量数据库；“有引用”也不等于“引用支持结论”。

In [ ]:
DOCS=[{'id':r['id'],'text':r['disease']+' '+r['microbe']} for r in STUDIES]
def rag_search(question):
    terms=set(re.findall(r'[a-z]+|[\u4e00-\u9fff]',question.lower()))
    scores=[(len(terms&set(re.findall(r'[a-z]+|[\u4e00-\u9fff]',d['text'].lower()))),d) for d in DOCS]
    return [d for n,d in sorted(scores,key=lambda x:-x[0]) if n>0]
print(rag_search('MDD Faecalibacterium'))
print('仅演示关键词检索，不代表医学证据或语义理解')

## 8 · 多智能体协作

**核心问题：**多Agent是不是必须购买三个GPT账号？

多智能体首先是一种任务分工方式：研究员检索资料，审查员判断证据是否合规，报告员将已经审核的数据整理成结果。三个角色可以运行在同一Python进程里，也可以未来再分别连接模型。只有明确职责、状态交接与审查边界，增加Agent数量才可能产生价值。

**操作步骤：**
1. 运行EvidenceResearcher取得资料
2. 交由EvidenceReviewer检查记录合法性
3. 由EvidenceWriter生成包含来源ID的报告

**预期输出：**能区分单个模型多角色提示、多个独立执行器和真正多模型系统。

**误区：**增加角色会增加成本和错误传播；审查步骤必须有可检验的规则。

In [ ]:
def researcher(q):return rag_search(q)
def reviewer(records):return [r for r in records if r['id'].startswith('DEMO')]
def writer(records):return '\n'.join(r['id']+': '+r['text'] for r in records) if records else '证据不足，拒绝回答'
print(writer(reviewer(researcher('MDD Faecalibacterium'))))

## 9 · 评估、安全与日志

**核心问题：**能生成答案，是否就代表Agent可靠？

科研Agent需要评估任务成功率、工具调用准确率、证据覆盖、误引率、拒答质量与可复现性。程序日志应记录动作、参数类别、耗时和错误，而不记录密钥。还要把检索到的网页、论文与提示词明确隔离，避免把外部资料中的恶意指令当成系统命令。

**操作步骤：**
1. 运行内置断言检查工具白名单和检索结果
2. 观察模拟提示注入如何被当成不可信资料
3. 写入JSONL轨迹并读取复核

**预期输出：**得到一组通过/失败的检查结果，以及结构化日志文件。

**误区：**只检查JSON结构正确不能证明科研内容真实；提示词隔离仍需要权限、审批和输出校验配合。

In [ ]:
checks={'tool_allowlist':'search_studies' in TOOLS,'forbidden_tool':'delete_all' not in TOOLS,'demo_label':all(x['id'].startswith('DEMO') for x in STUDIES)}
for name,ok in checks.items():print('PASS' if ok else 'FAIL',name)
assert all(checks.values())
with (OUTPUT/'trace.jsonl').open('a',encoding='utf-8') as f:f.write(json.dumps(checks)+'\n')

## 10 · MCP与外部工具协议

**核心问题：**Function Calling和MCP究竟有什么区别？

Function Calling描述模型如何提出工具调用；MCP提供客户端发现工具、读取资源、与外部工具服务通信的标准。通俗说，前者是模型提出‘我想做什么’，后者是外部能力如何用一致接口接入。课程先模拟工具描述与Client调用，帮助你看清Host、Client和Server的职责。

**操作步骤：**
1. 列出模拟的MCP工具元数据
2. 选择search_studies并通过执行器调用
3. 与上一节本地工具白名单比较

**预期输出：**能说清楚Host、Client、MCP Server、Tool四者的关系。

**误区：**教学模拟不等于实现完整MCP协议；接入真实服务需要权限、传输与版本兼容性检查。

In [ ]:
mcp_tool={'name':'search_studies','description':'查询教学数据','inputSchema':{'type':'object','properties':{'query':{'type':'string'}}}}
print('教学模拟：',json.dumps(mcp_tool,ensure_ascii=False))
print('模拟调用结果：',TOOLS['search_studies']('MDD'))
print('此例不是正式MCP SDK或MCP Server')

## 11 · 接入PubMed真实文献

**核心问题：**如何从模拟数据切换到真实研究信息？

使用NCBI E-utilities可在授权范围内查询PubMed记录。首次试验只获取PMID等元数据，不批量下载受版权保护的全文，也不把检索关键词匹配结果直接当成因果证据。真实资料必须与前面虚构的DEMO研究记录隔离，保持可追溯来源。

**操作步骤：**
1. 阅读pubmed_esearch函数并确认它使用HTTPS
2. 联网后手动打开RUN_PUBMED开关
3. 检索有限数量的PMID并记录查询式

**预期输出：**获得真实PMID列表；如网络不可用则清楚显示跳过，而不是伪造结果。

**误区：**PubMed需要网络并遵守NCBI使用规范；返回PMID不意味着已阅读论文全文。

In [ ]:
RUN_PUBMED=False
if RUN_PUBMED:
    from urllib.parse import urlencode
    from urllib.request import urlopen
    url='https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?'+urlencode({'db':'pubmed','term':'gut microbiome depression','retmode':'json','retmax':3})
    with urlopen(url,timeout=15) as response:print(json.load(response)['esearchresult']['idlist'])
else:print('默认关闭真实PubMed联网请求')

## 12 · 接入真实大语言模型

**核心问题：**真正让LLM决定什么时候调用函数，需要哪些代码？

在本节中，程序向Responses API声明只读工具的JSON Schema，模型可以返回函数调用请求，Python通过允许列表执行后，把调用结果连同关联标识传回模型。为防止误用或意外费用，联网默认关闭，需要自行配置模型和API Key，并设置最大调用轮数。关闭开关时只运行离线流程。

**操作步骤：**
1. 检查ENABLE_LIVE_LLM默认是False
2. 理解LIVE_TOOL_SCHEMAS中的函数名、参数和strict设置
3. 配置自己的API Key后手动打开开关，比较模型选工具与规则选工具

**预期输出：**在已授权且计费可用时看到真实模型的工具调用轨迹；未启用时明确跳过。

**误区：**API Key不能提交到GitHub；模型调用可能产生费用，且需要支持函数调用的可用模型。

In [ ]:
ENABLE_LIVE_LLM=False
if ENABLE_LIVE_LLM:
    import os
    from openai import OpenAI
    assert os.getenv('OPENAI_API_KEY'),'先配置环境变量 OPENAI_API_KEY'
    client=OpenAI()
    MODEL=os.getenv('OPENAI_MODEL','gpt-5-mini')
    schema=[{'type':'function','name':'search_studies','description':'查询虚构教学记录','parameters':{'type':'object','properties':{'query':{'type':'string'}},'required':['query'],'additionalProperties':False},'strict':True}]
    input_items=[{'role':'user','content':'请调用工具查询MDD教学记录，说明它们是虚构的。'}]
    for round_idx in range(3):
        response=client.responses.create(model=MODEL,input=input_items,tools=schema)
        input_items+=response.output
        calls=[x for x in response.output if x.type=='function_call']
        if not calls:
            print(response.output_text);break
        for c in calls:
            if c.name!='search_studies':raise ValueError('未经允许的工具')
            arguments=json.loads(c.arguments)
            if set(arguments)!={'query'} or not isinstance(arguments['query'],str):raise ValueError('参数无效')
            data=search_studies(arguments['query'])
            input_items.append({'type':'function_call_output','call_id':c.call_id,'output':json.dumps(data,ensure_ascii=False)})
else:print('LLM默认关闭：安装openai SDK并自备API Key后才能启用，可能产生费用')

## 13 · 多模态与Transformers选学

**核心问题：**看论文图表是否只需换个Prompt？

文本Agent处理的是字符串或结构化信息；多模态任务还需要图像输入、特定模型接口以及图表解析评估。课程演示本地文件加载与Hugging Face Transformers的可选模型调用，默认关闭下载和推理，避免无意中消耗大量磁盘、显存和网络额度。

**操作步骤：**
1. 识别本地图片路径并确认文件存在
2. 只在需要时开启图像加载演示
3. 在环境支持时再试可选Transformers模型

**预期输出：**能够区分文件读取、OCR、视觉推理和科学图表理解，不混淆不同能力。

**误区：**图表识别结果必须与原图坐标轴、图例和统计方法逐项对照。

In [ ]:
RUN_TRANSFORMERS=False
if RUN_TRANSFORMERS:
    from transformers import pipeline
    print(pipeline('sentiment-analysis')('This is a learning experiment.'))
else:print('跳过可能下载大模型的Transformers实验')

## 14 · 把Agent封装成API并部署

**核心问题：**Jupyter里能跑的程序怎么变成网页功能？

实际部署时，Notebook更适合研发与验证；长期提供服务通常要把稳定函数封装成FastAPI接口，让React前端通过HTTPS发请求。服务端负责管理凭据、访问权限、数据库连接和计算任务。课程生成一个本机只读API示例，不宣称它已经具备公开生产部署所需的认证、限流和监控。

**操作步骤：**
1. 运行生成demo_api.py的代码单元
2. 安装fastapi与uvicorn并在本机启动服务
3. 访问/docs检查接口，再理解云服务器和域名如何指向后端

**预期输出：**能理解浏览器→HTTP API→Python函数→JSON响应的运行路径。

**误区：**不要把未经鉴权的演示接口直接开放公网，更不能在网页JavaScript中放API Key。

In [ ]:
api_code='from fastapi import FastAPI\napp=FastAPI(title="教学科研API")\n@app.get("/health")\ndef health():return {"status":"ok"}\n@app.get("/studies")\ndef studies():return [{"id":"DEMO01","disease":"MDD"}]\n'
(OUTPUT/'demo_api.py').write_text(api_code,encoding='utf-8')
print('生成:',OUTPUT/'demo_api.py')
print('终端：python -m pip install fastapi uvicorn')
print('终端：cd agent_lab_output && uvicorn demo_api:app --reload')
print('打开 http://127.0.0.1:8000/docs')

## 15 · 微生物科研助手综合挑战

**核心问题：**从一个研究问题到带证据的报告怎样闭环？

毕业练习整合问题输入、疾病识别、文献检索、证据审查、报告生成和审计记录。整个离线版本仍基于教学模拟数据，目的是训练你对每个环节的控制能力。未来可把检索替换为PubMed真实文献，把摘要替换为LLM，并加入规范化抽取、元分析和人工审核。

**操作步骤：**
1. 运行MicrobiomeTeachingAgent处理一个疾病问题
2. 检查报告中的记录ID、检索范围和局限说明
3. 改变问题，比较新旧报告与工具轨迹

**预期输出：**生成可查看的教学报告及操作日志，能够说明哪些部分真实执行、哪些是模板或规则。

**误区：**不要把模拟分析报告作为真实疾病生物标志物结论；正式科研必须接入可核查来源与独立评估。

In [ ]:
def teaching_agent(question):
    evidence=reviewer(researcher(question))
    report={'question':question,'evidence_ids':[x['id'] for x in evidence],'answer':writer(evidence),'disclaimer':'虚构教学数据，不作为科研证据'}
    (OUTPUT/'report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    return report
print(json.dumps(teaching_agent('MDD Faecalibacterium'),ensure_ascii=False,indent=2))

## 后续路线

先掌握工具白名单、检索证据与任务轨迹，再连接真实PubMed及模型接口。运行任何真实LLM调用前阅读费用、安全、隐私和科研可复现性要求。